# NOWRI - publish to GitHub (no analysis needed)

This notebook only **uploads your finished project**. You do not need to re-run any analysis.

You need three things:
1. `nowri_repo_v0.1.1.zip` (the latest one from me)
2. the reports zip you downloaded from the main notebook (cell 8), named like `nowri_reports...zip`
3. a **fresh GitHub token** (classic; tick **repo** and **workflow**; 7 days). Paste **only the token**, nothing else, into the hidden box.

Run the three cells in order.

### 1. Upload the two zip files (select both in the file picker)

In [ ]:
import os, zipfile
from google.colab import files

os.chdir("/content")      # Colab saves uploads in the current folder
up = files.upload()      # choose BOTH files: nowri_repo_v0.1.1.zip and your nowri_reports...zip
repo, reps = [], []
for f in up:             # decide by what is INSIDE each zip, not by its name
    names = zipfile.ZipFile(f).namelist()
    if any(n.startswith("nowri/") for n in names):
        repo.append(f)
    elif any(n.startswith("reports/") for n in names):
        reps.append(f)
if not repo:
    raise SystemExit("I did not find the code zip (nowri_repo_v0.1.1.zip). Please run this cell again and choose it.")
zipfile.ZipFile(repo[0]).extractall("/content")
os.chdir("/content/nowri")
for r in reps:
    zipfile.ZipFile("/content/" + r).extractall("/content/nowri")      # adds the reports/ folders from your run
print("code zip:", repo[0], "| reports zip(s):", reps)
print("reports folders:", sorted(d for d in os.listdir("reports") if os.path.isdir(os.path.join("reports", d))))

### 2. Check what will be uploaded

In [ ]:
# Everything below is what will go to GitHub. There must be no .gpkg, .zip or other data files.
!cd /content/nowri && find . -type f -not -path "./.git/*" | sort

### 3. Upload to GitHub
If GitHub already holds a different version, this cell shows what is there and then replaces it with this one.
It refuses to upload if any data file is present, and it never prints your token.

In [ ]:
import base64, getpass, os, subprocess, sys
REPO = "Anas-Umar/nigerian-oil-water-risk-intelligence"
os.chdir("/content/nowri")
SECRETS = []          # anything in here is blanked out of every message this cell prints

def clean(text):
    for s in SECRETS:
        text = text.replace(s, "***")
    return text

def sh(*args, label, check=True):
    r = subprocess.run(list(args), capture_output=True, text=True)
    print(clean(r.stdout + r.stderr), end="")
    if check and r.returncode:
        raise SystemExit("STOPPED at step: " + label)
    return r

if not os.path.exists(".git"):
    sh("git", "init", "-b", "main", label="create local repository")
subprocess.run(["git", "remote", "remove", "origin"], capture_output=True)
sh("git", "remote", "add", "origin", f"https://github.com/{REPO}.git", label="set GitHub address")
sh("git", "config", "user.name", "Anas Umar Aliyu", label="set name")
sh("git", "config", "user.email", "Anas-Umar@users.noreply.github.com", label="set email")
sh("git", "add", "-A", label="stage files")
sh(sys.executable, "tools/check_no_data.py", "--staged", label="data policy check")   # stops if a data file slipped in
sh("git", "commit", "-m", "NOWRI v0.1.1", label="commit", check=False)               # 'nothing to commit' is fine

token = getpass.getpass("Paste ONLY the GitHub token (starts with ghp_), then press Enter: ").strip()
if not token.startswith(("ghp_", "github_pat_")) or len(token) < 30 or len(token.split()) != 1:
    raise SystemExit("That is not a token (it must be ONE line starting with ghp_). Nothing was sent anywhere.")
b64 = base64.b64encode(f"x-access-token:{token}".encode()).decode()
SECRETS += [token, b64]
auth = ["git", "-c", f"http.extraheader=Authorization: Basic {b64}"]

r = sh(*auth, "push", "-u", "origin", "main", label="push", check=False)
if r.returncode:
    print("\nGitHub already holds a different version of 'main'. Here is what is there now:")
    sh(*auth, "fetch", "origin", label="read GitHub's version")
    sh("git", "log", "--oneline", "-5", "origin/main", label="show GitHub history", check=False)
    sh("git", "ls-tree", "--name-only", "origin/main", label="show GitHub files", check=False)
    print("\nReplacing it with the version from this Colab session ...")
    sh(*auth, "push", "--force-with-lease=main:origin/main", "-u", "origin", "main", label="replace GitHub's version")
SECRETS.clear(); del token, b64, auth
print("\nUPLOADED to https://github.com/" + REPO)

### 4. Check it worked
Open https://github.com/Anas-Umar/nigerian-oil-water-risk-intelligence in a **private/incognito window**. You should see
`src`, `tests`, `reports`, `docs`, `LICENSE`, `DATA_POLICY.md` and a README that starts with the results table.
Then open the **Actions** tab (tests should turn green) and **delete your token** on GitHub (Settings, Developer settings,
Personal access tokens).